[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chung-I/MiRA_training_course_2026/blob/main/notebooks/pyg_node_classification.ipynb)

# Node Classification with GNNs (PyG Tutorial)

This notebook trains a GCN on the Cora citation network and compares it with an MLP baseline.
Section 5 sweeps the number of GCN layers to show over-smoothing.

## 1. Setup

In [ ]:
# On Colab, uncomment the next line:
# !pip install -q torch_geometric

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

PyTorch: 2.11.0+cu128
CUDA: True

## 2. Load the Cora dataset

Cora is a citation network. Each node is a paper with a 1,433-dimensional bag-of-words feature vector. Each edge is a citation. The label is one of 7 research topics. Only 140 nodes (5.2%) have training labels.

In [ ]:
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

print(f'Nodes: {data.num_nodes}')
print(f'Edges: {data.num_edges}')
print(f'Features per node: {data.num_node_features}')
print(f'Classes: {dataset.num_classes}')
print(f'Train / Val / Test: {data.train_mask.sum().item()} / {data.val_mask.sum().item()} / {data.test_mask.sum().item()}')

Nodes: 2708
Edges: 10556
Features per node: 1433
Classes: 7
Train / Val / Test: 140 / 500 / 1000

## 3. MLP baseline

An MLP uses the bag-of-words features but ignores the citation graph. Each node is classified independently.

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_channels, hidden_channels)
        self.lin2 = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x):
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.lin2(x)

accs = []
for seed in range(5):
    torch.manual_seed(seed)
    model = MLP(data.num_node_features, 16, dataset.num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    model.train()
    for epoch in range(200):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data.x)[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
    model.eval()
    pred = model(data.x).argmax(dim=1)
    acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    accs.append(acc)

print(f'MLP test accuracy (mean of 5 seeds): {sum(accs)/len(accs)*100:.1f}%')

MLP test accuracy (mean of 5 seeds): 54.7%

## 4. 2-layer GCN

A GCN replaces `Linear` with `GCNConv`, which aggregates each node's features with its neighbors through the citation edges.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)

accs = []
for seed in range(5):
    torch.manual_seed(seed)
    model = GCN(data.num_node_features, 16, dataset.num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    model.train()
    for epoch in range(200):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
    model.eval()
    pred = model(data.x, data.edge_index).argmax(dim=1)
    acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    accs.append(acc)

print(f'GCN test accuracy (mean of 5 seeds): {sum(accs)/len(accs)*100:.1f}%')

GCN test accuracy (mean of 5 seeds): 80.7%

The GCN reaches 80.7% vs the MLP's 54.7%. The 26-point gap comes from the citation graph: GCN propagates label information from the 140 labeled nodes along citation edges, while the MLP classifies each node from its features alone.

## 5. Over-smoothing: accuracy vs number of layers

Each GCN layer averages a node's features with its neighbors. After enough layers, every node in a connected component holds the same vector (Li et al., AAAI 2018). The table below sweeps the layer count from 1 to 8.

In [ ]:
class DeepGCN(torch.nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch, num_layers):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        if num_layers == 1:
            self.convs.append(GCNConv(in_ch, out_ch))
        else:
            self.convs.append(GCNConv(in_ch, hid_ch))
            for _ in range(num_layers - 2):
                self.convs.append(GCNConv(hid_ch, hid_ch))
            self.convs.append(GCNConv(hid_ch, out_ch))

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=0.5, training=self.training)
        return self.convs[-1](x, edge_index)

print(f'{"Layers":>6}  {"Test Acc":>8}')
print('-' * 18)
for num_layers in [1, 2, 3, 4, 6, 8]:
    accs = []
    for seed in range(5):
        torch.manual_seed(seed)
        model = DeepGCN(data.num_node_features, 16, dataset.num_classes, num_layers)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
        model.train()
        for epoch in range(200):
            optimizer.zero_grad()
            loss = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()
        model.eval()
        pred = model(data.x, data.edge_index).argmax(dim=1)
        acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        accs.append(acc)
    print(f'{num_layers:>6}  {sum(accs)/len(accs)*100:>7.1f}%')

Layers  Test Acc
------------------
     1     75.1%
     2     80.7%
     3     78.0%
     4     76.8%
     6     74.4%
     8     61.6%

Accuracy peaks at 2 layers (80.7%) and drops to 61.6% at 8 layers. Over-smoothing and the difficulty of training deep GCNs without residual connections both contribute to the collapse. On Cora, 2 layers is the optimum.